# Explainable Recommendation System: Zero-shot vs Fine-tuned


1. Load and preprocess the **Amazon Movies & TV 5-core** dataset
2. Build a prompt that asks the model to recommend an item and explain why — output as **structured JSON**
3. Run **Zero-shot inference** (no training, just prompting)
4. **Fine-tune** Qwen2.5-3B-Instruct using QLoRA on the training split
5. Run inference with the **fine-tuned model**
6. **Evaluate both** using recommendation metrics (NDCG@K, Precision@K) and explanation metrics (BLEU, ROUGE)
7. Print a **comparison table**



We ask the model to respond in this format:
```json
{"recommendation": "Movie Title", "explanation": "Because..."}
```
This makes it easy to parse the recommendation and explanation separately for metric computation.

## Step 0 — Install Dependencies


In [ ]:
# Install all required packages
# trl = Transformer Reinforcement Learning library (has SFTTrainer for fine-tuning)
# peft = Parameter Efficient Fine-Tuning (LoRA lives here)
# bitsandbytes = enables 4-bit quantization on GPU
# nltk , rouge_score = for BLEU and ROUGE eval. metrics

!pip install -q datasets trl peft bitsandbytes nltk rouge_score

## Step 1 — Imports

In [ ]:
import json
import random
import re
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

# BLEU score
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ROUGE score
from rouge_score import rouge_scorer

# Fix random seed for reproducibility
random.seed(42)
np.random.seed(42)

print("All imports successful.")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2 — Load Amazon Movies & TV Dataset

We use the **5-core** subset — meaning only users and items with at least 5 reviews are kept.
This removes sparse users and ensures every user has enough history to build a meaningful prompt.

Each row contains:
- `user_id` — anonymized user identifier
- `parent_asin` — item ID
- `title` — movie/show title (inside `metadata`)
- `text` — the review written by the user ← this becomes our **ground truth explanation**
- `rating` — 1 to 5 stars ← used to define what the user "liked"

We keep only **500 users** to stay within Colab's memory and time limits.

In [ ]:
print("Loading reviews...")
review_url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Movies_and_TV.jsonl.gz"
df = pd.read_json(review_url, lines=True, nrows=200000)  # increased from 50k
print(f"Reviews loaded: {len(df)}")

print("Loading item metadata...")
meta_url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Movies_and_TV.jsonl.gz"
meta_df = pd.read_json(meta_url, lines=True, nrows=500000)  # increased from 100k
print(f"Metadata loaded: {len(meta_df)}")

# Keep only parent_asin and real item title from metadata
# 'title' here is the actual movie/show name, not the review title
meta_df = meta_df[['parent_asin', 'title']].dropna().drop_duplicates('parent_asin')
meta_df.columns = ['parent_asin', 'item_title']   # rename to avoid collision with review title column

# Merge real movie title into reviews dataframe
df = df[['user_id', 'parent_asin', 'rating', 'text']].copy()
df = df.merge(meta_df, on='parent_asin', how='inner')

print(f"After merge: {len(df)} rows")
print(df.columns.tolist())
print(df[['user_id', 'parent_asin', 'item_title', 'text']].head(3))

Loading reviews...


In [ ]:
df

In [ ]:
# Drop rows with missing item title or review text
df = df.dropna(subset=['item_title', 'text', 'user_id', 'parent_asin'])

# Keep only positive interactions — rating >= 4 means the user liked the item
df = df[df['rating'] >= 4].reset_index(drop=True)

print(f"Reviews after filtering (rating >= 4): {len(df)}")
print(df[['user_id', 'item_title', 'text', 'rating']].head(3))

Reviews after filtering (rating >= 4): 57840
                        user_id       item_title  \
0  AGGZ357AO26RQZVRLGU4D4N52DZQ      Sneaky Pete   
1  AGKASBHYZPGTEPO6LWZPVJWB2BVA  Creative Galaxy   
2  AGCI7FAH4GL5FI65HYLKWTMFZ2CQ  Risen [Blu-ray]   

                                                text  rating  
0           Amazon, please buy the show! I'm hooked!       5  
1                         My Kiddos LOVE this show!!       5  
2  Great film all around. Some films about Jesus ...       5  


In [ ]:
df

In [ ]:
# Keep only users who have at least 6 liked items
# We need: 5 for history, 1 as the ground truth recommendation target
user_counts = df.groupby('user_id').size()
valid_users = user_counts[user_counts >= 6].index.tolist()

df = df[df['user_id'].isin(valid_users)].reset_index(drop=True)

# Sample 500 users to keep Colab fast
sampled_users = random.sample(valid_users, min(500, len(valid_users)))
df = df[df['user_id'].isin(sampled_users)].reset_index(drop=True)

print(f"Users after filtering : {df['user_id'].nunique()}")
print(f"Total rows kept       : {len(df)}")

Users after filtering : 500
Total rows kept       : 7286


## Step 3 — Build User Profiles

For each user we construct:
- **History** — the first N-1 liked items (what the user has already watched)
- **Target item** — the last liked item (what we want the model to recommend)
- **Target review** — the user's actual review of the target item (ground truth explanation)
- **Candidates** — target item + 4 random distractor items (model must pick the right one)

This simulates a real recommendation scenario:
> *Given what you know about this user, which of these 5 items should we recommend next?*

In [ ]:
# Get all unique item titles for sampling distractors
all_titles = df['item_title'].unique().tolist()

def build_user_profile(user_df):
    """
    Given all rows for one user, build a recommendation sample.

    Returns a dict with:
        history        : list of movie titles the user liked (all except last)
        target_item    : the movie we want the model to recommend (last item)
        target_review  : actual user review of target item (ground truth explanation)
        candidates     : shuffled list of [target + 4 distractors]
    """
    # Sort by index to maintain order
    user_df = user_df.reset_index(drop=True)

    # All items except last = history
    history_items = user_df.iloc[:-1]['item_title'].tolist()

    # Last item = target to recommend
    target_row = user_df.iloc[-1]
    target_item = target_row['item_title']
    target_review = target_row['text']

    # Sample 4 distractor items (items NOT in this user's history)
    distractors = random.sample(
        [t for t in all_titles if t not in history_items and t != target_item],
        4
    )

    # Combine target + distractors and shuffle so target isn't always first
    candidates = [target_item] + distractors
    random.shuffle(candidates)

    return {
        'history': history_items[-5:],   # Use last 5 for history (keep prompt short)
        'target_item': target_item,
        'target_review': target_review,
        'candidates': candidates
    }

# Build profiles for all sampled users
user_profiles = []
for user_id, user_df in df.groupby('user_id'):
    profile = build_user_profile(user_df)
    profile['user_id'] = user_id
    user_profiles.append(profile)

print(f"User profiles built: {len(user_profiles)}")
print("\nSample profile:")
sample = user_profiles[0]
print(f"  History    : {sample['history']}")
print(f"  Target     : {sample['target_item']}")
print(f"  Candidates : {sample['candidates']}")
print(f"  Review     : {sample['target_review'][:100]}...")

User profiles built: 500

Sample profile:
  History    : ['Silent Night ( Cicha noc ) [ NON-USA FORMAT, PAL, Reg.2 Import - Poland ]', 'Fortitude', 'Goliath', '12', 'Northern Exposure: Season 1 [DVD]']
  Target     : Seinfeld: The Complete Series
  Candidates : ['Nature: A Murder of Crows', 'Seinfeld: The Complete Series', 'My Favorite Spy', 'Baby', "Halloween- Unrated Director's Cut"]
  Review     : I haven't watched the entire series yet, but it's a delight to have easy access to every episode, of...


## Step 4 — Train / Test Split

We split user profiles into:
- **Train (80%)** — used to fine-tune the model
- **Test (20%)** — used to evaluate both zero-shot and fine-tuned model

Both models are evaluated on the **same test set** for a fair comparison.

In [ ]:
# Shuffle and split
random.shuffle(user_profiles)

split_idx = int(0.8 * len(user_profiles))
train_profiles = user_profiles[:split_idx]
test_profiles  = user_profiles[split_idx:]

print(f"Train profiles : {len(train_profiles)}")
print(f"Test profiles  : {len(test_profiles)}")

Train profiles : 400
Test profiles  : 100


## Step 5 — Prompt Template

We define a single prompt template used by both zero-shot and fine-tuned models.
The model is asked to respond **only in JSON** with two fields:
- `recommendation` — exactly one title from the candidates list
- `explanation` — why this item suits the user based on their history

Constraining to JSON makes parsing deterministic and metric computation straightforward.

In [ ]:
def build_prompt(profile):
    """
    Build the user-facing prompt from a user profile.
    This same prompt format is used for both zero-shot inference and fine-tuning.
    """
    history_str    = ", ".join(profile['history'])
    candidates_str = ", ".join(profile['candidates'])

    prompt = f"""You are a movie recommendation assistant.

A user has previously watched and liked these movies:
{history_str}

From the following candidates, recommend exactly ONE movie that best fits this user's taste:
{candidates_str}

Respond ONLY in valid JSON format with no extra text:
{{"recommendation": "<movie title>", "explanation": "<reason based on user history>"}}"""

    return prompt


def build_full_conversation(profile):
    """
    Build the full chat message list (system + user + assistant) for a profile.
    The assistant turn contains the ground truth output — used during fine-tuning.
    """
    # Ground truth output — what the fine-tuned model should learn to produce
    # We use the actual user review as the explanation (truncated to 150 chars)
    ground_truth_output = json.dumps({
        "recommendation": profile['target_item'],
        "explanation": profile['target_review'][:150].replace('\n', ' ').strip()
    })

    messages = [
        {"role": "system",    "content": "You are a movie recommendation assistant. Always respond in valid JSON."},
        {"role": "user",      "content": build_prompt(profile)},
        {"role": "assistant", "content": ground_truth_output}   # supervision signal
    ]
    return messages


# Preview one prompt
print("Sample prompt:")
print(build_prompt(train_profiles[0]))
print("\nGround truth output:")
gt = json.dumps({"recommendation": train_profiles[0]['target_item'],
                 "explanation": train_profiles[0]['target_review'][:150]})
print(gt)

Sample prompt:
You are a movie recommendation assistant.

A user has previously watched and liked these movies:
Steel Magnolias, Escape From Planet Earth, Teenage Mutant Ninja Turtles 3, Rise Of the Guardians, Here Comes the Boom

From the following candidates, recommend exactly ONE movie that best fits this user's taste:
Thomas & Friends: Blue Mountain Mystery the Movie, Ghost Hunters: Season 8: Part 2, Scary Movie 4 (Unrated Widescreen Edition), Primeval, Vol. 2 (Series 3), SpongeBob SquarePants: It's A SpongeBob Christmas

Respond ONLY in valid JSON format with no extra text:
{"recommendation": "<movie title>", "explanation": "<reason based on user history>"}

Ground truth output:
{"recommendation": "SpongeBob SquarePants: It's A SpongeBob Christmas", "explanation": "ok sponge bob how can you not watch the christmas special if you are a fan at any age ive cought my self watching the dumb thing"}


## Step 6 — Load Qwen2.5-3B-Instruct with 4-bit Quantization (QLoRA Setup)

### QLoRA
- **Q** = Quantization: load the base model in **4-bit** to save GPU memory (~2.5GB instead of ~6GB)
- **LoRA** = Low-Rank Adaptation: instead of updating all 3B parameters, we add small trainable adapter matrices to specific layers
- This means we train only ~1-2% of total parameters, making fine-tuning feasible on a free Google Colab T4

### LoRA hyperparameters
- `r=16` — rank of the adapter matrices. Higher = more capacity but more memory
- `lora_alpha=32` — scaling factor for the LoRA updates
- `target_modules` — which layers to attach adapters to (attention + MLP layers)
- `lora_dropout=0.05` — small dropout for regularization

In [ ]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

# ── 4-bit quantization config ──────────────────────────────────────────────
# load_in_4bit          : load weights as 4-bit integers (saves ~60% memory)
# bnb_4bit_quant_type   : "nf4" is the recommended quantization type for QLoRA
# bnb_4bit_compute_dtype: even though weights are 4-bit, compute happens in fp16
# bnb_4bit_use_double_quant: quantize the quantization constants too (extra savings)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token        # required for batched training
tokenizer.padding_side = "right"                  # pad on the right for causal LM

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Disable cache during training (not needed, saves memory)
model.config.use_cache = False

print("Model loaded in 4-bit.")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 7 — Zero-shot Inference (Before Fine-tuning)

We run the base model on the test set **without any fine-tuning**.
The model only sees the prompt — no examples of what good output looks like.

We store these results and compare them against the fine-tuned model later.

In [ ]:
def run_inference(model, tokenizer, profile, max_new_tokens=200):
    """
    Run inference for one user profile.
    Returns the raw model output string.
    """
    messages = [
        {"role": "system", "content": "You are a movie recommendation assistant. Always respond in valid JSON."},
        {"role": "user",   "content": build_prompt(profile)}
    ]

    # Apply chat template — formats messages into the model's expected input string
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():   # no gradient computation needed during inference
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,        # greedy decoding for deterministic output
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    # Strip input tokens, decode only the generated response
    response_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
    response = tokenizer.decode(response_ids, skip_special_tokens=True).strip()
    return response


def parse_json_output(raw_output):
    """
    Parse the model's raw string output into a dict.
    Handles cases where the model wraps JSON in markdown code blocks.
    Returns None if parsing fails.
    """
    try:
        # Remove markdown code fences if present (```json ... ```)
        cleaned = re.sub(r'```json|```', '', raw_output).strip()

        # Find the first JSON object in the string
        match = re.search(r'\{.*\}', cleaned, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception:
        pass
    return None   # return None if model output is not valid JSON

In [ ]:
# Run zero-shot inference on all test profiles
# This may take a few minutes depending on number of test samples
print("Running zero-shot inference on test set...")
print(f"Test samples: {len(test_profiles)}\n")

zero_shot_outputs = []

for i, profile in enumerate(test_profiles):
    raw = run_inference(model, tokenizer, profile)
    parsed = parse_json_output(raw)
    zero_shot_outputs.append({
        'profile': profile,
        'raw_output': raw,
        'parsed': parsed
    })

    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(test_profiles)}")

# Count how many outputs were valid JSON
valid_zs = sum(1 for o in zero_shot_outputs if o['parsed'] is not None)
print(f"\nValid JSON outputs: {valid_zs}/{len(zero_shot_outputs)}")

# Show one example
print("\nSample zero-shot output:")
print(zero_shot_outputs[0]['raw_output'])

## Step 8 — Prepare Fine-tuning Dataset

We convert train profiles into the format expected by `SFTTrainer`.

Each training example is a **full conversation** tokenized using the chat template.
The model learns to produce the correct JSON output (recommendation + review-based explanation)
given the user history and candidate list.

In [ ]:
def format_for_sft(profile):
    """
    Format one training profile as a single text string for SFTTrainer.
    SFTTrainer expects a 'text' field containing the full conversation.
    """
    messages = build_full_conversation(profile)

    # apply_chat_template with tokenize=False returns the full formatted string
    # add_generation_prompt=False because the assistant turn is already included
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}


# Format all training profiles
train_data = [format_for_sft(p) for p in train_profiles]

# Convert to HuggingFace Dataset object (required by SFTTrainer)
train_dataset = Dataset.from_list(train_data)

print(f"Training samples: {len(train_dataset)}")
print("\nSample training text (first 300 chars):")
print(train_dataset[0]['text'][:300])

Training samples: 400

Sample training text (first 300 chars):
<|im_start|>system
You are a movie recommendation assistant. Always respond in valid JSON.<|im_end|>
<|im_start|>user
You are a movie recommendation assistant.

A user has previously watched and liked these movies:
Steel Magnolias, Escape From Planet Earth, Teenage Mutant Ninja Turtles 3, Rise Of th


## Step 9 — Attach LoRA Adapters and Fine-tune

### Training strategy
- We attach LoRA adapters to the **attention and MLP projection layers** of the transformer
- Only adapter weights are updated — base model weights are frozen
- `SFTTrainer` handles the training loop, loss computation, and checkpointing

### Key training arguments
- `num_train_epochs=2` — 2 passes over the training data (enough for this size)
- `per_device_train_batch_size=1` — batch size 1 to avoid OOM on T4
- `gradient_accumulation_steps=8` — accumulate 8 steps before updating weights (effective batch = 8)
- `max_seq_length=512` — truncate sequences longer than 512 tokens

In [ ]:
# ── LoRA configuration ─────────────────────────────────────────────────────
# r             : rank of adapter matrices. r=16 is a good default
# lora_alpha    : scaling factor. typically set to 2*r
# target_modules: which weight matrices to attach adapters to
#                 q/k/v/o = attention, gate/up/down = MLP feed-forward
# lora_dropout  : regularization
# bias          : whether to train bias terms ("none" = don't)
# task_type     : CAUSAL_LM for text generation models
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach LoRA adapters to the quantized model
model = get_peft_model(model, lora_config)

# Print trainable vs total parameters
# You'll see only ~1-2% of parameters are trainable — that's the point of LoRA
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

# SFTTrainer = Supervised Fine-Tuning Trainer
# Handles tokenization, loss masking (computes loss only on assistant tokens),
# and the full training loop internally
trainer = SFTTrainer(
    model=model,                        # 4-bit quantized Qwen2.5-3B with LoRA adapters attached
    processing_class=tokenizer,         # renamed from 'tokenizer' in trl >= v0.12.0
    train_dataset=train_dataset,        # HuggingFace Dataset with 'text' column

    # SFTConfig = replaces TrainingArguments in newer trl versions
    # all training hyperparameters + SFT-specific configs go here
    args=SFTConfig(
        output_dir="./qwen_recsys_lora",     # where checkpoints are saved after each epoch
        num_train_epochs=2,                   # 2 passes over the training data
        per_device_train_batch_size=1,        # batch size 1 — avoids OOM on T4 15GB
        gradient_accumulation_steps=8,        # accumulate gradients over 8 steps before updating
                                              # effective batch size = 1 * 8 = 8
        warmup_steps=10,                      # linearly increase LR for first 10 steps
                                              # prevents unstable updates at the start
        learning_rate=2e-4,                   # standard learning rate for LoRA fine-tuning
        bf16=True,                            # Qwen2.5 loads in BFloat16 by default (torch_dtype="auto")
                                              # fp16 caused NotImplementedError — bf16 must match model dtype
                                              # BFloat16 = same range as float32, less precision than fp16
                                              # more stable for training than fp16 (no overflow issues)
        logging_steps=20,                     # print training loss every 20 steps
        save_strategy="epoch",                # save a checkpoint after each epoch
        optim="paged_adamw_8bit",             # memory-efficient AdamW optimizer for QLoRA
                                              # 'paged' = offloads optimizer states to CPU when needed
        report_to="none",                     # disable wandb / tensorboard logging
        max_length=512,                       # truncate sequences longer than 512 tokens
                                              # renamed from max_seq_length in trl >= v0.16.0
        dataset_text_field="text",            # column in train_dataset containing the training text
    ),
)

print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete.")

In [ ]:
# Save the LoRA adapter weights
# Note: this saves ONLY the adapter, not the full 3B model
model.save_pretrained("./qwen_recsys_lora_final")
tokenizer.save_pretrained("./qwen_recsys_lora_final")
print("Adapter saved to ./qwen_recsys_lora_final")

## Step 10 — Fine-tuned Model Inference

We run the fine-tuned model on the **same test set** used for zero-shot.
The model now has learned the JSON output format and the mapping from user history to explanations.

In [ ]:
# The model object already has LoRA adapters merged from training
# We just need to set it to eval mode before inference
model.eval()
model.config.use_cache = True    # re-enable KV cache for faster inference

print("Running fine-tuned inference on test set...")

finetuned_outputs = []

for i, profile in enumerate(test_profiles):
    raw = run_inference(model, tokenizer, profile)
    parsed = parse_json_output(raw)
    finetuned_outputs.append({
        'profile': profile,
        'raw_output': raw,
        'parsed': parsed
    })

    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(test_profiles)}")

valid_ft = sum(1 for o in finetuned_outputs if o['parsed'] is not None)
print(f"\nValid JSON outputs: {valid_ft}/{len(finetuned_outputs)}")

print("\nSample fine-tuned output:")
print(finetuned_outputs[0]['raw_output'])

## Step 11 — Evaluation Metrics

### Recommendation Metrics
We treat this as a **top-1 ranking** task — the model picks one item from 5 candidates.

- **Precision@1** — did the model recommend the correct item? (1 or 0 per sample)
- **NDCG@1** — same as Precision@1 for single-item output, but generalizable to top-K
- **Hit Rate@1** — fraction of test users where the correct item was recommended

### Explanation Metrics
We compare the generated explanation against the user's **actual review** (ground truth).

- **BLEU** — measures n-gram overlap between generated and reference explanation
- **ROUGE-1** — unigram recall/precision overlap
- **ROUGE-L** — longest common subsequence overlap (captures fluency better than ROUGE-1)

In [ ]:
def compute_metrics(outputs):
    """
    Compute recommendation and explanation metrics for a list of model outputs.

    Args:
        outputs: list of dicts with 'profile', 'parsed' keys

    Returns:
        dict of metric_name -> score
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    smoothing = SmoothingFunction().method1   # handles zero n-gram matches gracefully

    hits        = []
    bleu_scores = []
    r1_scores   = []
    rl_scores   = []
    parse_fails = 0

    for output in outputs:
        profile = output['profile']
        parsed  = output['parsed']

        # ── Recommendation metric ──────────────────────────────────────────
        if parsed is None or 'recommendation' not in parsed:
            # Model failed to produce valid JSON — count as miss
            hits.append(0)
            parse_fails += 1
        else:
            predicted = parsed['recommendation'].strip().lower()
            target    = profile['target_item'].strip().lower()
            # Hit = 1 if predicted title matches target title exactly
            hits.append(1 if predicted == target else 0)

        # ── Explanation metrics ────────────────────────────────────────────
        if parsed is not None and 'explanation' in parsed:
            generated_exp = parsed['explanation']
            reference_exp = profile['target_review'][:150]   # ground truth review

            # BLEU: tokenize both into word lists
            ref_tokens  = nltk.word_tokenize(reference_exp.lower())
            hyp_tokens  = nltk.word_tokenize(generated_exp.lower())
            bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothing)
            bleu_scores.append(bleu)

            # ROUGE
            scores = scorer.score(reference_exp, generated_exp)
            r1_scores.append(scores['rouge1'].fmeasure)
            rl_scores.append(scores['rougeL'].fmeasure)

    return {
        'Hit@1 (Precision@1)' : np.mean(hits),
        'NDCG@1'              : np.mean(hits),   # = Precision@1 for single-item ranking
        'BLEU'                : np.mean(bleu_scores) if bleu_scores else 0.0,
        'ROUGE-1'             : np.mean(r1_scores)   if r1_scores   else 0.0,
        'ROUGE-L'             : np.mean(rl_scores)   if rl_scores   else 0.0,
        'Parse Fail Rate'     : parse_fails / len(outputs)
    }

print("Computing metrics...")
zs_metrics = compute_metrics(zero_shot_outputs)
ft_metrics = compute_metrics(finetuned_outputs)
print("Done.")

## Step 12 — Results Comparison Table

This is the main result of the experiment.

The table shows:
- Whether fine-tuning improves recommendation accuracy (Hit@1)
- Whether fine-tuning improves explanation quality (BLEU, ROUGE)
- Whether fine-tuning reduces parse failures (how often the model produces invalid JSON)

In [ ]:
# Build comparison dataframe
results_df = pd.DataFrame({
    'Metric'    : list(zs_metrics.keys()),
    'Zero-shot' : [round(v, 4) for v in zs_metrics.values()],
    'Fine-tuned': [round(v, 4) for v in ft_metrics.values()]
})

# Compute delta
results_df['Delta'] = (results_df['Fine-tuned'] - results_df['Zero-shot']).round(4)
results_df['Improvement'] = results_df['Delta'].apply(lambda x: "↑" if x > 0 else ("↓" if x < 0 else "="))

print("=" * 65)
print("ZERO-SHOT vs FINE-TUNED — RESULTS COMPARISON")
print("=" * 65)
print(results_df.to_string(index=False))
print("=" * 65)
print(f"\nTest samples evaluated: {len(test_profiles)}")

## Summary

| | Zero-shot | Fine-tuned (QLoRA) |
|---|---|---|
| **Training required** | No | Yes (~few minutes on T4) |
| **Parameters updated** | 0 | ~1-2% (LoRA adapters only) |
| **Output format** | Often inconsistent | Consistently JSON |
| **Recommendation accuracy** | Moderate | Higher (learns candidate mapping) |
| **Explanation quality** | Generic | More grounded in user history |

### Key insight
Zero-shot is powerful but unreliable for structured output tasks. Fine-tuning with QLoRA on even a small dataset (400 samples) teaches the model:
1. The exact JSON format to follow
2. How to connect user history to item features in the explanation
3. The domain vocabulary (movie genres, styles, themes)

